# Assignment 2: Question 2

Copyies and augments logic from provided notebook. (Ran into an error running the first cell in the provided notebook)

## Setup

In [17]:
from google.colab import drive

REPO = 'ift_6135_representation_learning'
REPO_URL = f'https://github.com/trvslhlt/{REPO}.git'
MOUNTPOINT = '/content/gdrive'
BRANCH = 'assignment_2'
PROJECT_PATH = (
    '{}/MyDrive/education/universitie_de_montreal/'
    'courses/2026_winter/IFT_6135_B_representation_learning/colab_storage/assignment_2/'
).format(MOUNTPOINT)
PROJECT_DIR = '/content/project_dir'

drive.mount(MOUNTPOINT)

# create a symlink for easier access
# !rm -rf "{SYMLINK_DIR}"
!ln -s -f -n "{PROJECT_PATH}" "{PROJECT_DIR}"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [14]:
# move to the project directory going forward
%cd "{PROJECT_DIR}"

/content/gdrive/MyDrive/education/universitie_de_montreal/courses/2026_winter/IFT_6135_B_representation_learning/colab_storage/assignment_2


In [19]:
%%bash -s "$REPO" "$REPO_URL" "$BRANCH"
# clone repo, checkout branch, and pull most recent commits
git clone $2
cd $1
git checkout $3 && git pull

Your branch is up to date with 'origin/assignment_2'.
Already up to date.


fatal: destination path 'ift_6135_representation_learning' already exists and is not an empty directory.
Already on 'assignment_2'


In [21]:
%%bash -s "$REPO"
cd $1
pip install uv
pwd
# uv sync

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.4/23.4 MB 74.1 MB/s eta 0:00:00
/content/gdrive/MyDrive/education/universitie_de_montreal/courses/2026_winter/IFT_6135_B_representation_learning/colab_storage/assignment_2/ift_6135_representation_learning


This notebook is intended to produce the plots and figures for the report on Problem 1 of the practical. You should not run this notebook in Google Colab until you have finished constructing the correct solutions for transformer_solution.py and gru_solution.py

This notebook provides some limited commentary on several HuggingFace Features and toolage. You will use HuggingFace Datasets to load the Yelp Polarity dataset for sentiment analysis. The notebook will define a Bert tokenizer, collate functions, and then train and evaluate several models using the HuggingFace utilities mentioned above. Remember, the most crucial part here is running the experiments for the report.

In [22]:
%matplotlib inline
# %load_ext autoreload
# - ModuleNotFoundError: No module named 'imp'
# - WARNING: Upgrading ipython, ipykernel, tornado, prompt-toolkit, pyzmq can
#     cause your runtime to repeatedly crash or behave in unexpected ways and is not
#     recommended. If your runtime won't connect or execute code, you can reset it
#     with "Disconnect and delete runtime" from the "Runtime" menu.
# %autoreload 2

### Link assignment folder & install requirements
Enter the path to the assignment folder in your Google Drive
If you run this notebook locally or on a cluster (i.e. not on Google Colab)
you can delete this cell which is specific to Google Colab.

In [28]:
import sys
import os
import shutil
import warnings

# folder = "" #@param {type:"string"}
# !ln -Ts "$folder" /content/assignment 2> /dev/null
# !cp gdrive/MyDrive/Assignment2/transformer_solution.py .
# !cp gdrive/MyDrive/Assignment2/gru_solution.py .

ASSIGNMENT_DIR = os.path.join(PROJECT_DIR, REPO, 'assignments', 'assignment_2')

# Add the assignment folder to Python path
if ASSIGNMENT_DIR not in sys.path:
  sys.path.insert(0, ASSIGNMENT_DIR)

# Check if CUDA is available
import torch
if not torch.cuda.is_available():
  warnings.warn('CUDA is not available.')

/tmp/ipykernel_367/2382440144.py:20: UserWarning: CUDA is not available.
  warnings.warn('CUDA is not available.')


In [31]:
import matplotlib.pyplot as plt
import urllib.request
import time
import os
import json
import random
from typing import List, Dict, Union, Optional, Tuple

from sklearn.metrics import f1_score
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Optimizer, AdamW

from dataclasses import dataclass
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from datasets import Dataset, load_dataset, concatenate_datasets
from transformers import AutoModel, AutoTokenizer
from tokenizers import Tokenizer

from transformer_solution import Transformer, MultiHeadedAttention
from gru_solution import EncoderDecoder

In [32]:
def set_seed(seed: int = 0, device: torch.device = None):
    random.seed(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if device is not None and device.type == "cuda":
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    return rng

In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"--> Device selected: {device}")
rng = set_seed(seed=0, device=device)

--> Device selected: cpu


In [34]:
CACHE_DIR = os.path.join(PROJECT_DIR, 'data')
dataset_name = "yelp_polarity"
dataset_train_original = load_dataset(dataset_name, split="train", cache_dir=CACHE_DIR)
dataset_test_original = load_dataset(dataset_name, split="test", cache_dir=CACHE_DIR).shuffle(generator=rng)
dataset_test = dataset_test_original.select(range(1000))
dataset_train = concatenate_datasets([
    dataset_train_original,
    dataset_test_original.select(range(1000, len(dataset_test_original)))
])
print(f"{len(dataset_train)=}, {len(dataset_test)=}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

len(dataset_train)=597000, len(dataset_test)=1000


### 🔍 Quick look at the data
Lets have quick look at a few samples in our test set.

In [35]:
n_samples_to_see = 3
for i in range(n_samples_to_see):
  print("-"*30)
  print("title:", dataset_test[i]["text"])
  print("label:", dataset_test[i]["label"])

------------------------------
title: Best sweet potato fries I've ever had. Ever. \n\nJust the right amount of crispy with salt and sugar sprinkled on it and an addictive sweet sauce for dipping. Seriously these things were so good I went back the next night and ordered some to go after having a few drinks. \n\nI honestly don't even remember what my burger tasted like, but if you're staying at/near the Wynn definitely cross the street and check this place out.
label: 1
------------------------------
title: OK, next time I'm just trying the drinks.\n\nI didn't come here to drink, I came here for a nice date on a Friday night.  First off, I don't think you should introduce the night's specials by coming to the table 15 minutes after we have been seated and say \""have you guys seen this?\""  Oh, guess those are the specials.  Then we had to ask what the fish of the day was.  Needless to say, the service failed the place before anything was consumed.\n\nThe options on the menu were inter

# Experiments

### 1️. Tokenize the `text`
Tokenize the `text`portion of each sample (i.e. parsing the text to smaller chunks). Tokenization can happen in many ways; traditionally, this was done based on the white spaces. With transformer-based models, tokenization is performed based on the frequency of occurrence of "chunk of text". This frequency can be learned in many different ways. However the most common one is the [**wordpiece**](https://arxiv.org/pdf/1609.08144v2.pdf) model.
> The wordpiece model is generated using a data-driven approach to maximize the language-model likelihood
of the training data, given an evolving word definition. Given a training corpus and a number of desired
tokens $D$, the optimization problem is to select $D$ wordpieces such that the resulting corpus is minimal in the
number of wordpieces when segmented according to the chosen wordpiece model.

Under this model:
1. Not all things can be converted to tokens depending on the model. For example, most models have been pretrained without any knowledge of emojis. So their token will be `[UNK]`, which stands for unknown.
2. Some words will be mapped to multiple tokens!
3. Depending on the kind of model, your tokens may or may not respect capitalization

In [36]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [37]:
input_sample = "Welcome to IFT6135. We now teach you 🤗(HUGGING FACE) Library :DDD."
tokenizer.tokenize(input_sample)

['welcome',
 'to',
 'if',
 '##t',
 '##6',
 '##13',
 '##5',
 '.',
 'we',
 'now',
 'teach',
 'you',
 '[UNK]',
 '(',
 'hugging',
 'face',
 ')',
 'library',
 ':',
 'dd',
 '##d',
 '.']